# Sistema Automatizado - Pruebas Atrasadas 2026

Este notebook:
1. Lee emails de Gmail automaticamente
2. Extrae datos con EmailParser
3. Valida limite de 2 pruebas
4. Inserta directo en Google Sheets 2026

---

In [1]:
!pip install gspread oauth2client email-reply-parser

    extract-msg (<=0.29.*)
                 ~~~~~~~^


In [14]:
import imaplib
import email
from email.header import decode_header
import gspread
import re
from typing import Dict
from datetime import datetime
import getpass

print("Librerias cargadas OK")

Librerias cargadas OK


## CONFIGURACION

In [15]:
# TU CONFIGURACION
EMAIL_COLEGIO = "crojas@colsanjavier.cl"

# Contraseña de aplicacion (te la pedira de forma segura)
print("Ingresa tu contrasena de aplicacion de Gmail:")
EMAIL_PASSWORD = getpass.getpass("Contrasena: ")

# ID de tu Google Sheets 2026
GOOGLE_SHEETS_ID = "1H9b-dxrMN0Guzj69fkk-oHbgtjdAbH1TCyfKnJIWRyc"

print("\nConfiguracion lista")

Ingresa tu contrasena de aplicacion de Gmail:


Contrasena:  ················



Configuracion lista


## EmailParser (el mismo que ya funciona)

In [40]:
class EmailParser:
    CURSO_A_CICLO = {
        '7°A': 'III°CICLO', '7°B': 'III°CICLO', '7°C': 'III°CICLO',
        '7ºA': 'III°CICLO', '7ºB': 'III°CICLO', '7ºC': 'III°CICLO',
        '1°A': 'III°CICLO', '1°B': 'III°CICLO', '1°C': 'III°CICLO',
        '1ºA': 'III°CICLO', '1ºB': 'III°CICLO', '1ºC': 'III°CICLO',
        'I°A': 'III°CICLO', 'I°B': 'III°CICLO', 'I°C': 'III°CICLO',
        'IºA': 'III°CICLO', 'IºB': 'III°CICLO', 'IºC': 'III°CICLO',
        '2°A': 'IV°CICLO', '2°B': 'IV°CICLO', '2°C': 'IV°CICLO',
        '2ºA': 'IV°CICLO', '2ºB': 'IV°CICLO', '2ºC': 'IV°CICLO',
        'II°A': 'IV°CICLO', 'II°B': 'IV°CICLO', 'II°C': 'IV°CICLO',
        'IIºA': 'IV°CICLO', 'IIºB': 'IV°CICLO', 'IIºC': 'IV°CICLO',
        '3°A': 'IV°CICLO', '3°B': 'IV°CICLO', '3°C': 'IV°CICLO',
        '3ºA': 'IV°CICLO', '3ºB': 'IV°CICLO', '3ºC': 'IV°CICLO',
        'III°A': 'IV°CICLO', 'III°B': 'IV°CICLO', 'III°C': 'IV°CICLO',
        'IIIºA': 'IV°CICLO', 'IIIºB': 'IV°CICLO', 'IIIºC': 'IV°CICLO',
        '4°A': 'IV°CICLO', '4°B': 'IV°CICLO', '4°C': 'IV°CICLO',
        '4ºA': 'IV°CICLO', '4ºB': 'IV°CICLO', '4ºC': 'IV°CICLO',
        'IV°A': 'IV°CICLO', 'IV°B': 'IV°CICLO', 'IV°C': 'IV°CICLO',
        'IVºA': 'IV°CICLO', 'IVºB': 'IV°CICLO', 'IVºC': 'IV°CICLO',
    }
    
    MESES = {'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
             'julio': 7, 'agosto': 8, 'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12}
    
    DIAS_SEMANA = {'lunes': 'LUNES', 'martes': 'MARTES', 'miercoles': 'MIERCOLES',
                   'miércoles': 'MIERCOLES', 'jueves': 'JUEVES', 'viernes': 'VIERNES',
                   'sabado': 'SABADO', 'sábado': 'SABADO'}
    
    def __init__(self, texto_email: str, asunto_email: str):
        self.texto = texto_email
        self.asunto = asunto_email
    
    def parsear(self) -> dict:
        # Nombre y curso - "Estimada familia de Catalina Rojas IV°A"
        patron_nombre = r'familia de\s+(.+?)\s+(IV°[A-C]|III°[A-C]|II°[A-C]|I°[A-C]|[1-4]°[A-C]|[1-7]°[A-C])'
        match_nombre = re.search(patron_nombre, self.texto, re.IGNORECASE)
        
        if match_nombre:
            nombre = match_nombre.group(1).strip()
            curso = match_nombre.group(2).strip()
        else:
            # Formato viejo
            patron_viejo = r'PRUEBA ATRASADA\s+(.+?)\s+(IV°[A-C]|III°[A-C]|[1-4]°[A-C])'
            match_viejo = re.search(patron_viejo, self.asunto, re.IGNORECASE)
            nombre = match_viejo.group(1).strip() if match_viejo else None
            curso = match_viejo.group(2).strip() if match_viejo else None
        
        # Asignatura - CON ASTERISCOS: "*Asignatura:* Ciencias"
        patron_asig = r'\*?Asignatura:\*?\s*(.+?)(?:\r\n|\n|$)'
        match_asig = re.search(patron_asig, self.texto, re.IGNORECASE)
        asignatura = match_asig.group(1).strip() if match_asig else None
        
        # Fecha - CON ASTERISCOS: "*Fecha de realización:* miércoles 3 de diciembre"
        patron_fecha = r'\*?Fecha de realizaci[oó]n:\*?\s*(\w+)\s+(\d+)\s+de\s+(\w+)'
        match_fecha = re.search(patron_fecha, self.texto, re.IGNORECASE)
        
        nombre_hoja = None
        if match_fecha:
            dia_semana = match_fecha.group(1).lower()
            dia = int(match_fecha.group(2))
            mes = match_fecha.group(3).lower()
            dia_formato = self.DIAS_SEMANA.get(dia_semana, dia_semana.upper())
            mes_num = self.MESES.get(mes, 0)
            nombre_hoja = f"{dia_formato} {dia:02d}{mes_num:02d}"
        
        # Duración - CON ASTERISCOS: "*Duración: 60 minutos.*"
        patron_dur = r'\*?[Dd]uraci[oó]n:\s*(\d+)\s*minutos?'
        match_dur = re.search(patron_dur, self.texto, re.IGNORECASE)
        duracion = f"{match_dur.group(1)} minutos" if match_dur else None
        
        return {
            'nombre_estudiante': nombre,
            'curso': curso,
            'ciclo': self.CURSO_A_CICLO.get(curso, 'DESCONOCIDO'),
            'asignatura': asignatura,
            'nombre_hoja': nombre_hoja,
            'duracion': duracion,
            'observaciones': f"El control tiene una duración de {duracion}." if duracion else "",
            'asistencia': ''
        }

print("EmailParser FINAL cargado")



EmailParser FINAL cargado


In [45]:
# PRUEBA DEL PARSER
texto_email = """*Estimada familia de  Catalina Rojas IV°A *

Junto con saludarles, informamos que su hijo debe realizar la siguiente
evaluación que tiene pendiente,

· *Asignatura:* Ciencias

· *Fecha de realización:* miércoles 3 de diciembre

· *Horario de inicio:* 15:30 hrs.

· *Duración: 60 minutos. *

· *Lugar:*  1°C"""

asunto_email = "Prueba atrasada "

# Probar el parser
parser = EmailParser(texto_email, asunto_email)
datos = parser.parsear()

print("RESULTADO DEL PARSER:")
print("=" * 60)
print(f"Nombre: {datos['nombre_estudiante']}")
print(f"Curso: {datos['curso']}")
print(f"Ciclo: {datos['ciclo']}")
print(f"Asignatura: {datos['asignatura']}")
print(f"Hoja: {datos['nombre_hoja']}")
print(f"Duración: {datos['duracion']}")

RESULTADO DEL PARSER:
Nombre: Catalina Rojas
Curso: IV°A
Ciclo: IV°CICLO
Asignatura: Ciencias
Hoja: MIERCOLES 0312
Duración: 60 minutos


## Conectar a Gmail

In [17]:
def conectar_gmail():
    try:
        imap = imaplib.IMAP4_SSL("imap.gmail.com")
        imap.login(EMAIL_COLEGIO, EMAIL_PASSWORD)
        print("Conectado a Gmail OK")
        return imap
    except Exception as e:
        print(f"ERROR conectando a Gmail: {e}")
        return None

# Probar conexion
imap = conectar_gmail()
if imap:
    # Solo hacer logout, no close (porque no abrimos ninguna carpeta)
    imap.logout()
    print("Conexion verificada y cerrada")


Conectado a Gmail OK
Conexion verificada y cerrada


## Conectar a Google Sheets

In [41]:
print("Conectando a Google Sheets...")

gc = gspread.oauth(
    credentials_filename='credentials.json',
    authorized_user_filename='token.json'
)

spreadsheet = gc.open_by_key(GOOGLE_SHEETS_ID)
print(f"Google Sheets abierto: {spreadsheet.title}")
print(f"Hojas disponibles: {len(spreadsheet.worksheets())}")

Conectando a Google Sheets...
Google Sheets abierto: Nomina estudiantes pruebas atrasadas 2026
Hojas disponibles: 77


## Funcion para leer emails

In [42]:
def leer_emails_pruebas(limite=10):
    imap = conectar_gmail()
    if not imap:
        return []
    
    try:
        imap.select("INBOX")
        status, messages = imap.search(None, 'SUBJECT "PRUEBA ATRASADA"', 'UNSEEN')
        email_ids = messages[0].split()
        
        print(f"Emails no leidos: {len(email_ids)}")
        
        emails = []
        for email_id in email_ids[-limite:]:
            status, msg_data = imap.fetch(email_id, "(RFC822)")
            msg = email.message_from_bytes(msg_data[0][1])
            
            asunto = decode_header(msg["Subject"])[0][0]
            if isinstance(asunto, bytes):
                asunto = asunto.decode()
            
            cuerpo = ""
            if msg.is_multipart():
                for part in msg.walk():
                    if part.get_content_type() == "text/plain":
                        cuerpo = part.get_payload(decode=True).decode()
                        break
            else:
                cuerpo = msg.get_payload(decode=True).decode()
            
            emails.append({'asunto': asunto, 'cuerpo': cuerpo})
        
        imap.close()
        imap.logout()
        return emails
    
    except Exception as e:
        print(f"ERROR: {e}")
        return []

# Probar
emails = leer_emails_pruebas(limite=5)
print(f"\nEmails encontrados: {len(emails)}")
for i, e in enumerate(emails, 1):
    print(f"  {i}. {e['asunto']}")

Conectado a Gmail OK
Emails no leidos: 1

Emails encontrados: 1
  1. Prueba atrasada 


In [55]:
class ValidadorPruebas:
    """
    Valida que un estudiante no tenga más de 2 pruebas en una hoja de Google Sheets
    """
    
    def __init__(self, spreadsheet):
        self.spreadsheet = spreadsheet
    
    def normalizar_texto(self, texto):
        if not texto:
            return ""
        
        reemplazos = {
            'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
            'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
            'ñ': 'n', 'Ñ': 'N'
        }
        
        texto_normalizado = texto
        for viejo, nuevo in reemplazos.items():
            texto_normalizado = texto_normalizado.replace(viejo, nuevo)
        
        return texto_normalizado.strip().upper()
    
    def contar_pruebas_estudiante(self, nombre_hoja, nombre_estudiante, asignatura):
        try:
            worksheet = self.spreadsheet.worksheet(nombre_hoja)
            datos = worksheet.get_all_values()
            
            nombre_normalizado = self.normalizar_texto(nombre_estudiante)
            asignatura_normalizada = self.normalizar_texto(asignatura)
            
            num_pruebas = 0
            tiene_esta_asignatura = False
            
            for fila in datos[7:]:  
                if len(fila) >= 4:
                    nombre_en_fila = self.normalizar_texto(fila[2])
                    asignatura_en_fila = self.normalizar_texto(fila[3])
                    
                    if nombre_en_fila == nombre_normalizado:
                        num_pruebas += 1
                        if asignatura_en_fila == asignatura_normalizada:
                            tiene_esta_asignatura = True
            
            prueba_seria = num_pruebas + 1
            
            if num_pruebas >= 2:
                return {
                    'valido': False,
                    'num_pruebas': num_pruebas,
                    'prueba_seria': prueba_seria,
                    'tiene_esta_asignatura': tiene_esta_asignatura,
                    'mensaje': f'❌ RECHAZADO ({prueba_seria}/2) - LÍMITE EXCEDIDO'
                }
            elif tiene_esta_asignatura:
                return {
                    'valido': True,
                    'num_pruebas': num_pruebas,
                    'prueba_seria': prueba_seria,
                    'tiene_esta_asignatura': True,
                    'mensaje': f'✅ VÁLIDO ({prueba_seria}/2) - Reemplazará {asignatura}'
                }
            else:
                return {
                    'valido': True,
                    'num_pruebas': num_pruebas,
                    'prueba_seria': prueba_seria,
                    'tiene_esta_asignatura': False,
                    'mensaje': f'✅ VÁLIDO ({prueba_seria}/2) - Prueba {prueba_seria}'
                }
        
        except Exception as e:
            return {
                'valido': False,
                'num_pruebas': 0,
                'mensaje': f'❌ ERROR: {str(e)}'
            }

print("ValidadorPruebas cargado")

ValidadorPruebas cargado


In [59]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

class ResponderEmails:
    """
    Envía respuestas automáticas por email
    """
    
    def __init__(self, email_usuario, password):
        self.email_usuario = email_usuario
        self.password = password
    
    def enviar_confirmacion_inscripcion(self, destinatario, nombre_estudiante, curso, asignatura, fecha_hoja):
        """
        Envía email confirmando que la inscripción fue exitosa
        """
        try:
            # Crear mensaje
            msg = MIMEMultipart()
            msg['From'] = self.email_usuario
            msg['To'] = destinatario
            msg['Subject'] = f"✅ Inscripción exitosa - {nombre_estudiante} - {asignatura}"
            
            cuerpo = f"""
Estimado/a profesor/a:

La inscripción de la prueba atrasada ha sido procesada exitosamente:

- Estudiante: {nombre_estudiante}
- Curso: {curso}
- Asignatura: {asignatura}
- Fecha programada: {fecha_hoja}

El estudiante ha sido inscrito en el sistema.

Saludos cordiales,
Sistema Automatizado de Pruebas Atrasadas
            """
            
            msg.attach(MIMEText(cuerpo, 'plain'))
            
            # Enviar
            server = smtplib.SMTP_SSL('smtp.gmail.com', 465)
            server.login(self.email_usuario, self.password)
            server.send_message(msg)
            server.quit()
            
            return True
        
        except Exception as e:
            print(f"    ⚠️  Error al enviar confirmación: {e}")
            return False
    
    def enviar_rechazo_limite(self, destinatario, nombre_estudiante, curso, asignatura, num_pruebas):
        """
        Envía email informando que se rechazó por límite excedido
        """
        try:
            # Crear mensaje
            msg = MIMEMultipart()
            msg['From'] = self.email_usuario
            msg['To'] = destinatario
            msg['Subject'] = f"❌ Inscripción rechazada - {nombre_estudiante} - Límite excedido"
            
            cuerpo = f"""
Estimado/a profesor/a:

La inscripción de la prueba atrasada NO pudo ser procesada:

- Estudiante: {nombre_estudiante}
- Curso: {curso}
- Asignatura: {asignatura}

MOTIVO: El estudiante ya tiene {num_pruebas} pruebas inscritas para esa fecha.
Según reglamento, el límite máximo es de 2 pruebas por día.

Por favor, coordine una fecha alternativa o comuníquese con el Director Academico o la Dirección de ciclo respectiva.

Saludos cordiales,

Claudio Rojas 
Dirección Academica
            """
            
            msg.attach(MIMEText(cuerpo, 'plain'))
            
            # Enviar
            server = smtplib.SMTP_SSL('smtp.gmail.com', 465)
            server.login(self.email_usuario, self.password)
            server.send_message(msg)
            server.quit()
            
            return True
        
        except Exception as e:
            print(f"    ⚠️  Error al enviar rechazo: {e}")
            return False

print("ResponderEmails cargado")

ResponderEmails cargado


## PROCESO COMPLETO AUTOMATIZADO

In [60]:
print("=" * 80)
print("PROCESANDO EMAILS CON VALIDACIÓN Y RESPUESTAS AUTOMÁTICAS")
print("=" * 80)

emails = leer_emails_pruebas(limite=10)

if not emails:
    print("\nNo hay emails nuevos para procesar")
else:
    # Crear validador y respondedor
    validador = ValidadorPruebas(spreadsheet)
    respondedor = ResponderEmails(EMAIL_COLEGIO, EMAIL_PASSWORD)
    
    for i, email_data in enumerate(emails, 1):
        print(f"\n[{i}/{len(emails)}] {email_data['asunto'][:50]}...")
        
        # Parsear
        parser = EmailParser(email_data['cuerpo'], email_data['asunto'])
        datos = parser.parsear()
        
        print(f"  Estudiante: {datos['nombre_estudiante']}")
        print(f"  Curso: {datos['curso']}")
        print(f"  Asignatura: {datos['asignatura']}")
        print(f"  Hoja: {datos['nombre_hoja']}")
        
        # VALIDAR
        validacion = validador.contar_pruebas_estudiante(
            datos['nombre_hoja'],
            datos['nombre_estudiante'],
            datos['asignatura']
        )
        
        print(f"  {validacion['mensaje']}")
        
        # Procesar según resultado
        if validacion['valido']:
            # INSCRIBIR
            try:
                worksheet = spreadsheet.worksheet(datos['nombre_hoja'])
                
                nueva_fila = [
                    datos['ciclo'],
                    datos['curso'],
                    datos['nombre_estudiante'],
                    datos['asignatura'],
                    datos['asistencia'],
                    datos['observaciones']
                ]
                
                worksheet.append_row(nueva_fila)
                print(f"  ✅ INSCRITO en Google Sheets")
                
                # ENVIAR CONFIRMACIÓN
                # Nota: El email viene del remitente original
                # Por ahora enviamos a nosotros mismos como prueba
                respondedor.enviar_confirmacion_inscripcion(
                    EMAIL_COLEGIO,  # Cambiar por email del profesor cuando esté listo
                    datos['nombre_estudiante'],
                    datos['curso'],
                    datos['asignatura'],
                    datos['nombre_hoja']
                )
                print(f"  📧 Email de confirmación enviado")
                
            except Exception as e:
                print(f"  ❌ ERROR al insertar: {e}")
        
        else:
            # RECHAZAR
            print(f"  ⛔ NO SE INSCRIBE - Límite excedido")
            
            # ENVIAR RECHAZO
            respondedor.enviar_rechazo_limite(
                EMAIL_COLEGIO,  # Cambiar por email del profesor cuando esté listo
                datos['nombre_estudiante'],
                datos['curso'],
                datos['asignatura'],
                validacion['num_pruebas']
            )
            print(f"  📧 Email de rechazo enviado")
    
    print(f"\n=" * 80)
    print(f"COMPLETADO: {len(emails)} emails procesados")
    print(f"=" * 80)

PROCESANDO EMAILS CON VALIDACIÓN Y RESPUESTAS AUTOMÁTICAS
Conectado a Gmail OK
Emails no leidos: 1

[1/1] Prueba atrasada ...
  Estudiante: Catalina Rojas
  Curso: IV°A
  Asignatura: Lebguane
  Hoja: MIERCOLES 1803
  ❌ RECHAZADO (3/2) - LÍMITE EXCEDIDO
  ⛔ NO SE INSCRIBE - Límite excedido
  📧 Email de rechazo enviado

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
COMPLETADO: 1 emails procesados


In [64]:
import subprocess
import os

# Cambiar al directorio correcto
os.chdir('/Users/claudiorojas/Documents/Ciencia de Datos/Pruebas Atrasadas')

# Ejecutar el script
resultado = subprocess.run(
    ['python3', 'sistema_pruebas_automatizado.py'],
    capture_output=True,
    text=True
)

print(resultado.stdout)
if resultado.stderr:
    print("\n--- ERRORES ---")
    print(resultado.stderr)


EJECUCIÓN AUTOMÁTICA - 2026-02-15 21:48:23
✅ Conectado a Google Sheets
Emails no leidos: 0

No hay emails nuevos para procesar



In [63]:
import os

# Ver dónde estás trabajando
print("Directorio actual:")
print(os.getcwd())

print("\n\nArchivos en este directorio:")
for archivo in os.listdir('.'):
    print(f"  - {archivo}")

print("\n\n¿Existe sistema_pruebas_automatizado.py?")
existe = os.path.exists('sistema_pruebas_automatizado.py')
print(f"  {existe}")

if existe:
    print("\n  Ruta completa:")
    print(f"  {os.path.abspath('sistema_pruebas_automatizado.py')}")

Directorio actual:
/Users/claudiorojas/Documents/Ciencia de Datos/Pruebas Atrasadas


Archivos en este directorio:
  - Clave Gmail.png
  - Verificacion_Setup_Google.ipynb
  - Sistema_Automatizado_FINAL.ipynb
  - GUIA_VISUAL.md
  - JSON Aplicacion Pruebas atrasadas .rtf
  - Nomina estudiantes pruebas atrasadas 2025-2.numbers
  - token.json
  - Crear_Google_Sheets_2026_LIMPIO.ipynb
  - Preparacion_Gmail_API.ipynb
  - Sin título.numbers
  - Sistema_Pruebas_FINAL_CORREGIDO.ipynb
  - GUIA_SETUP_GOOGLE_CLOUD.md
  - Nomina estudiantes pruebas atrasadas 2025-2.xlsx
  - README.md
  - sistema_pruebas_automatizado.py
  - credentials.json
  - Sistema_Automatizado_FINAL.textClipping
  - .ipynb_checkpoints


¿Existe sistema_pruebas_automatizado.py?
  True

  Ruta completa:
  /Users/claudiorojas/Documents/Ciencia de Datos/Pruebas Atrasadas/sistema_pruebas_automatizado.py


---

## LISTO

Sistema funcionando:
- Lee emails automaticamente
- Extrae datos
- Inserta en Google Sheets 2026

### Para ejecutar regularmente:
1. Abre este notebook
2. Ejecuta todas las celdas
3. Procesara emails nuevos automaticamente